# Notebook Purpose

The purpose of this notebook is to replicate the random forest results.

## Imports

In [1]:
import time
import statistics
from pathlib import Path
from datetime import datetime, timezone, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import trange
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, accuracy_score

from experiment_helper import ExperimentHelper
from datasets.dataset_handler import DatasetHandler
from datasets.enums import FeatureType, DatasetSource

## Experiment Setup

In [2]:
experiment_id, experiment_dir, logger = ExperimentHelper.start_experiment("random_forest", Path("experiments") / "experiment_log")

Started experiment 'random_forest_20260926_114258'


## Attributes

In [3]:
handler = DatasetHandler()

SUPPORTED_FEATURE_TYPES = [FeatureType.ECFP]  # random forest requires fixed-length vector features

dataset_source = ExperimentHelper.prompt_enum_choice(logger, DatasetSource, "dataset source")
feature_type = ExperimentHelper.prompt_enum_choice(logger, FeatureType, "feature type", supported=SUPPORTED_FEATURE_TYPES)

Available dataset sources:
  0: tox21
  1: muv
  2: dude_gpcr


Select dataset source (name or index):  0


Selected dataset source: 'tox21'
Available feature types:
  0: ecfp
  1: dgl (not supported)
  2: dgl_with_bonds (not supported)


Select feature type (name or index):  0


Selected feature type: 'ecfp'


## Methods

### Save ROC/PRC Plot

Method to plot and save the ROC and Precision-Recall curves for a single round of an experiment run.

In [4]:
def save_roc_prc_plot(plots_dir, run_name, round_idx, fpr, tpr, precision, recall, roc, prc):
    plots_dir.mkdir(parents=True, exist_ok=True)

    fig, ax = plt.subplots()
    ax.plot(fpr, tpr, color="tab:blue")
    ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title(f"ROC Curve - {run_name} - Round {round_idx} (AUC = {roc:.3f})")
    fig.savefig(plots_dir / f"{run_name}_round{round_idx}_roc.png")
    plt.close(fig)

    fig, ax = plt.subplots()
    ax.plot(recall, precision, color="tab:orange")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.set_title(f"PRC Curve - {run_name} - Round {round_idx} (AP = {prc:.3f})")
    fig.savefig(plots_dir / f"{run_name}_round{round_idx}_prc.png")
    plt.close(fig)

### Build Result Row

Method to build a result row dict for a given run, sharing the common environment/config fields between the summary and per-round CSVs. Extra fields (e.g. metrics) are merged in via keyword arguments.

In [5]:
def build_result_row(dt_run_start, dataset_source_val, feature_type_val, no_pos, no_neg, target, **extra_fields):
    return {
        'EXPERIMENT_DATE': dt_run_start,
        'CPU': cpu,
        'CPU COUNT': logical_cpu_count,
        'GPU': gpu,
        'GPU RAM': gpu_ram_total_mb,
        'RAM': general_ram_gb,
        'CUDA': cuda_version,
        'DATASET SOURCE': dataset_source_val,
        'FEATURE TYPE': feature_type_val,
        'N_ESTIMATORS': n_estimators,
        'NUMBER POSITIVE': no_pos,
        'NUMBER NEGATIVE': no_neg,
        'TARGET': target,
        **extra_fields
    }

## Main Logic

### Device Information

In [6]:
device_info = ExperimentHelper.get_device_info(logger)
cpu = device_info["cpu"]
logical_cpu_count = device_info["logical_cpu_count"]
gpu = device_info["gpu"]
gpu_ram_total_mb = device_info["gpu_ram_total_mb"]
general_ram_gb = device_info["general_ram_gb"]
cuda_version = device_info["cuda_version"]

CPU: 'AMD Ryzen 5 3600 6-Core Processor'
Physical CPU Count: '6'
Logical CPU Count: '12'
CUDA Version: 'Not Available'
GPU: 'Not Available'
Available RAM: '26.8 GB'
GPU RAM: 'Not Available'


### Parameters

In [7]:
rounds = 20
randomseed = 12
n_estimators = 100
device = ExperimentHelper.set_global_seed(randomseed)

combinations = [
    [10, 10],
    [5, 10], 
    [1, 10], 
    [1, 5], 
    [1, 1]
]

save_columns = [
    'EXPERIMENT_DATE',
    'CPU',
    'CPU COUNT',
    'GPU',
    'GPU RAM',
    'RAM',
    'CUDA',
    'DATASET SOURCE',
    'FEATURE TYPE',
    'N_ESTIMATORS',
    'NUMBER POSITIVE',
    'NUMBER NEGATIVE',
    'TEST POSITIVE',
    'TEST NEGATIVE',
    'TARGET',
    'ACCURACY',
    'ROC',
    'PRC',
    'TRAINING TIME'
]

save_columns_rounds = [
    'EXPERIMENT_DATE',
    'CPU',
    'CPU COUNT',
    'GPU',
    'GPU RAM',
    'RAM',
    'CUDA',
    'DATASET SOURCE',
    'FEATURE TYPE',
    'N_ESTIMATORS',
    'NUMBER POSITIVE',
    'NUMBER NEGATIVE',
    'TEST POSITIVE',
    'TEST NEGATIVE',
    'TARGET',
    'ROUND',
    'SEED',
    'ACCURACY',
    'ROC',
    'PRC'
]

### Runs

In [8]:
# train tasks discarded: RF is a no-transfer baseline, fit per-target on the support set only
_, test_dfs = handler.load_train_test_set(dataset_source=dataset_source, feature_type=feature_type)
dataset_source_val = dataset_source.value
feature_type_val = feature_type.value

result_rows = [] # accumulates summary rows; built into a DataFrame once at the end
round_rows = [] # accumulates per-round rows; built into a DataFrame once at the end
results_csv = experiment_dir / f"{dataset_source_val}_{feature_type_val}.csv"
results_rounds_csv = experiment_dir / f"{dataset_source_val}_{feature_type_val}_rounds.csv"
plots_dir = experiment_dir / "plots" / dataset_source_val / feature_type_val

for no_pos, no_neg in combinations:
    for target in test_dfs.keys():
        dt_run_start = datetime.now(timezone.utc).strftime("%d/%m/%Y %H:%M:%S")
        logger.info(f"\nRunning random forest experiment for the '{dataset_source_val}' dataset with '{feature_type_val}' features, '{no_pos}' positives and '{no_neg}' negatives for '{target}' target")

        run_name = f"{target}_pos{no_pos}_neg{no_neg}"
        running_acc = []
        running_roc = []
        running_prc = []
        test_pos_count = None
        test_neg_count = None

        start_time = time.time()
        for r in trange(rounds):
            seed = randomseed + r
            test_df = test_dfs[target]

            support_neg = test_df[test_df['y'] == 0].sample(no_neg, random_state=seed)
            support_pos = test_df[test_df['y'] == 1].sample(no_pos, random_state=seed)

            train_data = pd.concat([support_neg, support_pos])
            test_data = test_df.drop(train_data.index)

            train_X, train_y = list(train_data['mol'].to_numpy()), train_data['y'].to_numpy(dtype=np.int16)
            test_X, test_y = list(test_data['mol'].to_numpy()), test_data['y'].to_numpy(dtype=np.int16)

            test_pos_count = int((test_y == 1).sum())
            test_neg_count = int((test_y == 0).sum())

            model = RandomForestClassifier(n_estimators=n_estimators, random_state=seed)
            model.fit(train_X, train_y)
            probs_y = model.predict_proba(test_X)
            preds_y = (probs_y[:, 1] >= 0.5).astype(int)

            acc = accuracy_score(test_y, preds_y)
            roc = roc_auc_score(test_y, probs_y[:, 1])
            prc = average_precision_score(test_y, probs_y[:, 1])

            fpr, tpr, _ = roc_curve(test_y, probs_y[:, 1])
            precision, recall, _ = precision_recall_curve(test_y, probs_y[:, 1])

            save_roc_prc_plot(plots_dir, run_name, r, fpr, tpr, precision, recall, roc, prc)

            running_acc.append(acc)
            running_roc.append(roc)
            running_prc.append(prc)

            round_rows.append(build_result_row(
                dt_run_start, dataset_source_val, feature_type_val, no_pos, no_neg, target,
                **{
                    'TEST POSITIVE': test_pos_count,
                    'TEST NEGATIVE': test_neg_count,
                    'ROUND': r,
                    'SEED': seed,
                    'ACCURACY': acc,
                    'ROC': roc,
                    'PRC': prc
                }
            ))

        end_time = time.time()
        duration = str(timedelta(seconds=(end_time - start_time)))

        rounds_acc = f"{statistics.mean(running_acc):.3f} \u00B1 {statistics.stdev(running_acc):.3f}"
        rounds_roc = f"{statistics.mean(running_roc):.3f} \u00B1 {statistics.stdev(running_roc):.3f}"
        rounds_prc = f"{statistics.mean(running_prc):.3f} \u00B1 {statistics.stdev(running_prc):.3f}"

        result_rows.append(build_result_row(
            dt_run_start, dataset_source_val, feature_type_val, no_pos, no_neg, target,
            **{
                'TEST POSITIVE': test_pos_count,
                'TEST NEGATIVE': test_neg_count,
                'ACCURACY': rounds_acc,
                'ROC': rounds_roc,
                'PRC': rounds_prc,
                'TRAINING TIME': duration
            }
        ))

result_df = pd.DataFrame(result_rows, columns=save_columns)
result_rounds_df = pd.DataFrame(round_rows, columns=save_columns_rounds)

logger.info(f"Saving experiment results to CSV at '{results_csv}'")
result_df.to_csv(results_csv, index=False)
logger.info(f"Saving per-round experiment results to CSV at '{results_rounds_csv}'")
result_rounds_df.to_csv(results_rounds_csv, index=False)


Running random forest experiment for the 'tox21' dataset with 'ecfp' features, '10' positives and '10' negatives for 'SR-HSE' target
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:05<00:00,  3.54it/s]

Running random forest experiment for the 'tox21' dataset with 'ecfp' features, '10' positives and '10' negatives for 'SR-MMP' target
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:05<00:00,  3.88it/s]

Running random forest experiment for the 'tox21' dataset with 'ecfp' features, '10' positives and '10' negatives for 'SR-p53' target
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████